In [36]:
import numpy as np
import json
import random
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, confusion_matrix


In [6]:
def calculate_greedy_cost(p_prev, p_curr, p_next, c=0):
    return abs(p_prev - p_curr)
# revise based on previous level

def calculate_planning_cost(p_prev, p_curr, p_next, c=0):

    dist_step_1 = abs(p_prev - p_curr)
    dist_step_2 = abs(p_curr - p_next)
    

    v1 = p_curr - p_prev
    v2 = p_next - p_curr
    
    switch_penalty = c if (v1 * v2 < 0) else 0
    
    return dist_step_1 + dist_step_2 + switch_penalty

In [7]:
def generate_levels(num_levels=50, screen_width=1, 
                    greedy_func=calculate_greedy_cost, 
                    planning_func=calculate_planning_cost, 
                    c=0,
                    degree_of_conflict=1.5):
    # generate levels wo regard to screen size, scale up based 
    # set doc to 1
    experiment_configs = []
    
    while len(experiment_configs) < num_levels:
        # level 1
        h1_entry = random.uniform(0.1, 0.9)
        
        # second level
        # sample two dist randomly, set near and far wo biasing
        #side = random.choice([-1, 1]) 
        h2_left_side = random.uniform(0, h1_entry)
        h2_right_side = random.uniform(h1_entry, 1)
        

        # third level
        h3_goal_x = random.uniform(h2_left_side, h2_right_side)
        if not (0 < h3_goal_x < screen_width):
            continue

        # costs
        cost_g_left = greedy_func(h1_entry, h2_left_side, h3_goal_x, c)
        cost_p_left = planning_func(h1_entry, h2_left_side, h3_goal_x, c)
        
        cost_g_right = greedy_func(h1_entry, h2_right_side, h3_goal_x, c)
        cost_p_right = planning_func(h1_entry, h2_right_side, h3_goal_x, c)

        #greedy_prefers_a = (cost_g_a * degree_of_conflict < cost_g_b)
        #planner_prefers_b = (cost_p_b * degree_of_conflict < cost_p_a)
        
        if True:
            trial = {
                "trial_id": len(experiment_configs) + 1,
                "levels": [
                    {"level": 1, "holes": [h1_entry]},
                    {"level": 2, "holes": [h2_left_side, h2_right_side]},
                    {"level": 3, "holes": [h3_goal_x]}
                ],
                "metadata": {
                    "left_greedy_cost": cost_g_left,
                    "right_greedy_cost": cost_g_right,
                    "left_planning_cost": cost_p_left,
                    "right_planning_cost": cost_p_right,
                    #"switch_cost_param": c,
                }
            }
            experiment_configs.append(trial)
            
    return experiment_configs

In [33]:
trials = generate_levels(num_levels=50000)
print(trials[1])


{'trial_id': 2, 'levels': [{'level': 1, 'holes': [0.2647826752117244]}, {'level': 2, 'holes': [0.0725430188805278, 0.5248949538198153]}, {'level': 3, 'holes': [0.524234215398817]}], 'metadata': {'left_greedy_cost': 0.19223965633119658, 'right_greedy_cost': 0.26011227860809094, 'left_planning_cost': 0.6439308528494857, 'right_planning_cost': 0.2607730170290893}}


In [34]:
def get_trial_features(trials):
    greedy_choice_is_left = []
    matrix_rows = []

    for trial in trials:
        meta = trial['metadata']
        levels = trial['levels']
        
        g_cost_L = meta['left_greedy_cost']
        g_cost_R = meta['right_greedy_cost']
        p_cost_L = meta['left_planning_cost']
        p_cost_R = meta['right_planning_cost']
        
        greedy_choice_is_left.append(1 if g_cost_L < g_cost_R else 0)

        matrix_rows.append([g_cost_L, g_cost_R, p_cost_L, p_cost_R])

    return np.array(greedy_choice_is_left).reshape(-1, 1), np.array(matrix_rows)

In [ ]:
data = get_trial_features(trials)
y = pd.DataFrame(data[0])
X = pd.DataFrame(data[1])
feature_names = ['Left Greedy Cost', 'Right Greedy Cost', 'Left Planning Cost', 'Right Planning Cost']

#model = sm.Logit(y, X).fit(disp = 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = LogisticRegression(solver='liblinear', random_state=0)
model.fit(X_train, y_train)

coefficients = pd.Series(model.coef_[0], index=feature_names)
print(coefficients)

Left Greedy Cost      -33.741753
Right Greedy Cost      33.736970
Left Planning Cost     -0.848978
Right Planning Cost     0.839411
dtype: float64


c:\Users\manik\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [39]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.2f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

Accuracy Score: 0.99
Confusion Matrix:
[[6193   32]
 [  32 6243]]
